# ML-00 XGBoost Native Categorical Run

`ml_inputs/ml_00_test/run_02_xgb_native_cat_15f`에 생성된 native categorical 입력을 사용해 ML-00 XGBoost 학습/validation 흐름을 실행한다.

## 실행 범위

- `fb_outputs` 검토 후 `ml_inputs`에 승인 배치된 `run_02_xgb_native_cat_15f` 입력을 사용한다.
- `encoding_manifest.json`을 `XGBTrainConfig`에 전달해 native categorical dtype을 복원한다.
- train split으로 모델을 학습하고 validation split에서 threshold와 metric을 산출한다.
- feature 생성이나 `fb_outputs -> ml_inputs` 복사는 수행하지 않는다.
- ML-00은 파이프라인 구성과 기준선 확보가 목적이며, 세부 하이퍼파라미터 튜닝은 ML-01 이후에서 진행한다.
- final test는 기본 실행하지 않는다. full validation artifact 확정 후 사용자가 `RUN_FINAL_TEST=True`로 전환해 실행한다.

## 실행 전 주의

- 입력 feature set은 `*_feature_contract_approve.csv`에서 `used_in_ml=TRUE`인 컬럼을 기준으로 한다.
- native categorical dtype 복원에는 같은 run directory의 `encoding_manifest.json`이 필요하다.
- threshold 선택은 validation set에서만 수행하고, test set은 `RUN_FINAL_TEST=True`일 때만 사용한다.
- 기본 overwrite는 `False`다. 같은 설정으로 재실행하려면 `MODEL_RUN_ID` 변경을 권장한다.

# 00_경로 및 환경설정

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

# ============================================================
# 0.1 Runtime Settings
# 실행 환경: ML 담당자 기준
# - Kernel          : Local WSL
# - Code repo       : local Git repo
# - Input artifacts : local Git repo ml/ml-00_baseline/ml_inputs/
# - Output artifacts: local Git repo ml/ml-00_baseline/ml_outputs/
# ============================================================
SEED = 42

# Root Paths: 다른 환경에서 실행할 경우 보통 이 경로만 수정한다.
LOCAL_REPO_ROOT = Path("/mnt/d/AML_git/Graph_AML").resolve()

# Input/Output Paths
ML_00_BASE_DIR = LOCAL_REPO_ROOT / "ml" / "ml-00_baseline"
ML_00_ML_INPUTS_DIR = ML_00_BASE_DIR / "ml_inputs"
ML_00_ML_OUTPUTS_DIR = ML_00_BASE_DIR / "ml_outputs"

# Module Code Paths: local Git repo에서 train_val_test 모듈을 import한다.
ML_00_MODULE_TVT_DIR = ML_00_BASE_DIR / "train_val_test"

# ============================================================
# 0.2 Path Validation
# ============================================================
def require_dir(path: Path, name: str) -> None:
    # 필수 디렉터리가 없으면 시작 단계에서 중단한다.
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"{name} is not a directory: {path}")

# 시작 단계에서 반드시 존재해야 하는 디렉터리만 검증한다.
for name, path in {
    "LOCAL_REPO_ROOT": LOCAL_REPO_ROOT,
    "ML_00_BASE_DIR": ML_00_BASE_DIR,
    "ML_00_ML_INPUTS_DIR": ML_00_ML_INPUTS_DIR,
    "ML_00_ML_OUTPUTS_DIR": ML_00_ML_OUTPUTS_DIR,
    "ML_00_MODULE_TVT_DIR": ML_00_MODULE_TVT_DIR,
}.items():
    require_dir(path, name)

# ============================================================
# 0.3 Import Path Setup
# ============================================================
# local train_val_test/ml_00_ml_*.py 모듈을 우선 import한다.
IMPORT_DIRS = [str(ML_00_MODULE_TVT_DIR)]
IMPORT_MODULES = ("ml_00_ml_utils", "ml_00_ml_io", "ml_00_ml_train", "ml_00_ml_val", "ml_00_ml_test")

# 중복 경로를 제거한 뒤 맨 앞에 삽입해 import 우선순서를 고정한다.
sys.path = [path for path in sys.path if path not in IMPORT_DIRS]
sys.path[:0] = IMPORT_DIRS

# 노트북 재실행 시 수정된 local 모듈을 다시 읽도록 import cache를 제거한다.
for module_name in IMPORT_MODULES:
    sys.modules.pop(module_name, None)

import ml_00_ml_utils
import ml_00_ml_io
import ml_00_ml_train
import ml_00_ml_val
import ml_00_ml_test

# train/validation/test 실행 전 난수 시드를 고정한다.
ml_00_ml_utils.set_seed(SEED)

# ============================================================
# 0.4 Configuration Summary
# ============================================================
print("=" * 80)
print("SEED                    :", SEED)
print("LOCAL_REPO_ROOT         :", LOCAL_REPO_ROOT)
print("ML_00_ML_INPUTS_DIR     :", ML_00_ML_INPUTS_DIR)
print("ML_00_ML_OUTPUTS_DIR    :", ML_00_ML_OUTPUTS_DIR)
print("ML_00_MODULE_TVT_DIR    :", ML_00_MODULE_TVT_DIR)
print("=" * 80)

SEED                    : 42
LOCAL_REPO_ROOT         : /mnt/d/AML_git/Graph_AML
ML_00_ML_INPUTS_DIR     : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs
ML_00_ML_OUTPUTS_DIR    : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs
ML_00_MODULE_TVT_DIR    : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/train_val_test


# 01_실행 설정

In [2]:
# ============================================================
# 실행 예시
# ============================================================
# 예시 1. Native categorical full train/validation 실행
# - encoding manifest가 있는 승인 입력으로 모델 학습 + validation threshold 선택
# - final test는 아직 실행하지 않음
#   RUN_TRAIN_AND_VALIDATION = True
#   SAMPLE_ROWS = None
#   RUN_FINAL_TEST = False
#
# 예시 2. 기존 full artifact로 final test만 실행
# - 같은 OUT_DIR에 model.pkl, feature_columns.json, train_summary.json, threshold.json이 있을 때만 실행
# - train/validation은 다시 돌리지 않고 final test만 수행
#   RUN_TRAIN_AND_VALIDATION = False
#   SAMPLE_ROWS = None
#   RUN_FINAL_TEST = True
#
# 주의:
# - RUN_ID는 fb_outputs 검토 후 ml_inputs에 승인 배치된 입력 묶음 식별자다.
# - MODEL_RUN_ID는 같은 입력으로 반복하는 모델 학습/검증 시도 식별자다.
# - RUN_KIND는 sample/full 구분용이다.
# - OVERWRITE_OUTPUTS=False이면 기존 산출물이 있을 때 덮어쓰지 않고 중단한다.

# ============================================================
# 산출물 구분
# ============================================================
# model.pkl                    -> train 결과
# feature_columns.json          -> train 결과
# train_summary.json            -> train 결과 + native categorical 설정 기록
# feature_importance.csv         -> train 결과, booster importance
# threshold.json                -> validation 결과
# metrics_val.json              -> validation 결과
# confusion_matrix_val.csv      -> validation 결과
# metrics_test.json             -> final test 결과
# confusion_matrix_test.csv     -> final test 결과

# ============================================================
# 1.1 Run identifiers
# ============================================================
# EXPORT_EXPERIMENT_ID / RUN_ID: 입력 산출물 폴더를 구성하는 식별자
# MODEL_RUN_ID: 같은 입력으로 반복하는 모델 학습/검증 run 식별자
EXPORT_EXPERIMENT_ID = "ml_00_test"
RUN_ID = "run_02_xgb_native_cat_15f"
MODEL_RUN_ID = "default_00"
""
RUN_KIND = "full"
ARTIFACT_PREFIX = f"{EXPORT_EXPERIMENT_ID}__{RUN_ID}"

# ============================================================
# 1.2 Input / output directories
# ============================================================
# ML_INPUT_DIR은 사람이 검토 후 승인 배치한 ML 입력 위치다.
# OUT_DIR은 train/validation/test artifact를 저장하는 최종 run directory다.
ML_INPUT_DIR = ML_00_ML_INPUTS_DIR / EXPORT_EXPERIMENT_ID / RUN_ID
ML_OUTPUT_DIR = ML_00_ML_OUTPUTS_DIR / EXPORT_EXPERIMENT_ID / RUN_ID
OUT_DIR = ML_OUTPUT_DIR / RUN_KIND / MODEL_RUN_ID

# ============================================================
# 1.3 Input artifact paths
# ============================================================
# train/val/test parquet와 승인 feature contract는 같은 ARTIFACT_PREFIX를 사용한다.
# ENCODING_MANIFEST_PATH는 native categorical dtype 복원과 검증에 사용된다.
TRAIN_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_train.parquet"
VAL_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_val.parquet"
TEST_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_test.parquet"
FEATURE_COLUMNS_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_feature_contract_approve.csv"
ENCODING_MANIFEST_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_encoding_manifest.json"

# ============================================================
# 1.4 Execution switches
# ============================================================
# SAMPLE_ROWS=None이면 full split을 사용한다. 숫자를 넣으면 smoke/debug 샘플 실행이다.
# final test는 full model + validation threshold 확정 후 명시적으로 True로 바꾼다.
SAMPLE_ROWS = None
RUN_TRAIN_AND_VALIDATION = True
RUN_FINAL_TEST = True
OVERWRITE_OUTPUTS = False

# ============================================================
# 1.5 XGBoost parameters
# ============================================================
# native categorical 활성화 여부는 encoding manifest에서 복원된 categorical feature 존재 여부로 train 모듈이 결정한다.
XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "min_child_weight": 1.0,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "gamma": 0.0,
    "early_stopping_rounds": 30,
    "n_jobs": -1,
}

# ============================================================
# 1.6 Input contract object
# ============================================================
# InputPaths에 encoding manifest를 포함해 train/val/test 단계가 같은 encoding 계약을 사용하게 한다.
INPUT_PATHS = ml_00_ml_io.InputPaths(
    train_path=TRAIN_PATH,
    val_path=VAL_PATH,
    test_path=TEST_PATH,
    feature_columns_path=FEATURE_COLUMNS_PATH,
    encoding_manifest_path=ENCODING_MANIFEST_PATH,
)

# ============================================================
# 1.7 Configuration preview
# ============================================================
print("=" * 80)
print("RUN_ID                 :", RUN_ID)
print("MODEL_RUN_ID           :", MODEL_RUN_ID)
print("TRAIN_PATH             :", TRAIN_PATH)
print("VAL_PATH               :", VAL_PATH)
print("FEATURE_COLUMNS_PATH   :", FEATURE_COLUMNS_PATH)
print("ENCODING_MANIFEST_PATH :", ENCODING_MANIFEST_PATH)
print("OUT_DIR                :", OUT_DIR)
print("=" * 80)

RUN_ID                 : run_02_xgb_native_cat_15f
MODEL_RUN_ID           : default_00
TRAIN_PATH             : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_native_cat_15f_Xy_train.parquet
VAL_PATH               : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_native_cat_15f_Xy_val.parquet
FEATURE_COLUMNS_PATH   : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_native_cat_15f_feature_contract_approve.csv
ENCODING_MANIFEST_PATH : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_native_cat_15f_encoding_manifest.json
OUT_DIR                : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_02_xgb_native_cat_15f/full/default_00


# 02_train / validation 실행

In [3]:
# 입력 parquet, 승인 feature CSV, encoding manifest가 모두 존재하는지 먼저 확인한다.
ml_00_ml_io.require_input_files(INPUT_PATHS, require_test=RUN_FINAL_TEST)
ml_00_ml_io.print_input_paths(INPUT_PATHS)

if RUN_TRAIN_AND_VALIDATION:
    # train_config는 train split, validation split, feature mask, native categorical manifest를 한 번에 묶는다.
    train_config = ml_00_ml_train.XGBTrainConfig(
        train_path=TRAIN_PATH,
        val_path=VAL_PATH,
        feature_columns_path=FEATURE_COLUMNS_PATH,
        output_dir=OUT_DIR,
        sample_rows=SAMPLE_ROWS,
        overwrite=OVERWRITE_OUTPUTS,
        seed=SEED,
        encoding_manifest_path=ENCODING_MANIFEST_PATH,
        **XGB_PARAMS,
    )
    train_result = ml_00_ml_train.train_xgb(train_config)

    # threshold 선택과 validation metric 산출은 validation split에서만 수행한다.
    val_config = ml_00_ml_val.ValidationConfig(
        val_path=VAL_PATH,
        output_dir=OUT_DIR,
        sample_rows=SAMPLE_ROWS,
        overwrite=OVERWRITE_OUTPUTS,
        encoding_manifest_path=ENCODING_MANIFEST_PATH,
    )
    val_result = ml_00_ml_val.validate_xgb(val_config)
    print("train_summary_path :", train_result.train_summary_path)
    print("threshold_path     :", val_result.threshold_path)
    print("metrics_path       :", val_result.metrics_path)
else:
    print("RUN_TRAIN_AND_VALIDATION=False: skipped")

train_path          : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_native_cat_15f_Xy_train.parquet
val_path            : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_native_cat_15f_Xy_val.parquet
test_path           : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_native_cat_15f_Xy_test.parquet
feature_columns_path: /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_native_cat_15f_feature_contract_approve.csv
encoding_manifest_path: /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_02_xgb_native_cat_15f/ml_00_test__run_02_xgb_native_cat_15f_encoding_manifest.json
train_summary_path : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_02_xgb_native_cat_15f/full/default_00/train_summary.json
t

# 03_final test 잠금

In [4]:
# final test는 기본 잠금 상태다. 확정된 full artifact가 있을 때만 RUN_FINAL_TEST=True로 전환한다.
if RUN_FINAL_TEST:
    # confirm_final_test=True를 명시해 test 평가가 의도된 실행임을 남긴다.
    test_config = ml_00_ml_test.TestConfig(
        test_path=TEST_PATH,
        output_dir=OUT_DIR,
        confirm_final_test=True,
        overwrite=OVERWRITE_OUTPUTS,
        encoding_manifest_path=ENCODING_MANIFEST_PATH,
    )
    test_result = ml_00_ml_test.test_xgb(test_config)
    print("metrics_path:", test_result.metrics_path)
else:
    print("RUN_FINAL_TEST=False: final test locked")

metrics_path: /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_02_xgb_native_cat_15f/full/default_00/metrics_test.json


# 04_artifact 요약

저장된 train/validation/final-test artifact를 다시 읽어 실제 학습에 사용된 feature 목록과 metric을 확인한다. 이 섹션은 평가를 새로 실행하지 않는다.

In [6]:
import json
from pathlib import Path

import pandas as pd


def _read_json(path: Path) -> dict:
    with Path(path).open(encoding="utf-8") as f:
        return json.load(f)


def _artifact_paths(out_dir: Path) -> dict[str, Path]:
    return {
        "model": out_dir / "model.pkl",
        "feature_columns": out_dir / "feature_columns.json",
        "train_summary": out_dir / "train_summary.json",
        "feature_importance": out_dir / "feature_importance.csv",
        "threshold": out_dir / "threshold.json",
        "metrics_val": out_dir / "metrics_val.json",
        "confusion_matrix_val": out_dir / "confusion_matrix_val.csv",
        "metrics_test": out_dir / "metrics_test.json",
        "confusion_matrix_test": out_dir / "confusion_matrix_test.csv",
    }


def _load_actual_feature_columns(paths: dict[str, Path]) -> list[str]:
    feature_columns_path = paths["feature_columns"]
    if feature_columns_path.exists():
        return list(_read_json(feature_columns_path)["feature_columns"])

    approve_table = pd.read_csv(FEATURE_COLUMNS_PATH, encoding="utf-8-sig", dtype={"used_in_ml": "string"})
    return approve_table.loc[approve_table["used_in_ml"] == "TRUE", "column_name"].astype(str).tolist()


def _build_feature_summary(paths: dict[str, Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    actual_features = _load_actual_feature_columns(paths)
    actual_df = pd.DataFrame({
        "feature_order": range(1, len(actual_features) + 1),
        "column_name": actual_features,
    })

    approve_table = pd.read_csv(FEATURE_COLUMNS_PATH, encoding="utf-8-sig", dtype={"used_in_ml": "string"})
    contract_cols = [
        "column_name",
        "used_in_ml",
        "feature_group",
        "dtype",
        "xgb_feature_type",
        "encoding",
        "source_column",
        "excluded_reason",
        "leakage_risk",
        "selection_note",
    ]
    contract_cols = [col for col in contract_cols if col in approve_table.columns]
    feature_summary = actual_df.merge(approve_table[contract_cols], on="column_name", how="left")

    if paths["feature_importance"].exists():
        importance = pd.read_csv(paths["feature_importance"])
        importance = importance.rename(columns={"feature": "column_name"})
        importance_cols = [
            "column_name",
            "rank_by_gain",
            "importance_gain",
            "importance_weight",
            "importance_cover",
        ]
        importance_cols = [col for col in importance_cols if col in importance.columns]
        feature_summary = feature_summary.merge(importance[importance_cols], on="column_name", how="left")

    manifest_features = []
    if ENCODING_MANIFEST_PATH.exists():
        manifest_features = list(_read_json(ENCODING_MANIFEST_PATH).get("feature_columns", []))

    approved_features = approve_table.loc[
        approve_table["used_in_ml"] == "TRUE", "column_name"
    ].astype(str).tolist()
    actual_set = set(actual_features)
    alignment_rows = []
    for feature_name in manifest_features:
        if feature_name not in actual_set:
            alignment_rows.append({
                "column_name": feature_name,
                "source": "manifest_only",
                "used_in_approve_contract": feature_name in approved_features,
                "used_in_trained_model": False,
            })
    for feature_name in approved_features:
        if feature_name not in actual_set:
            alignment_rows.append({
                "column_name": feature_name,
                "source": "approve_only",
                "used_in_approve_contract": True,
                "used_in_trained_model": False,
            })
    alignment_df = pd.DataFrame(alignment_rows)
    return feature_summary, alignment_df


def _build_run_summary(paths: dict[str, Path]) -> pd.DataFrame:
    if not paths["train_summary"].exists():
        return pd.DataFrame([{"status": "missing_train_summary", "path": str(paths["train_summary"])}])

    train_summary = _read_json(paths["train_summary"])
    threshold_payload = _read_json(paths["threshold"]) if paths["threshold"].exists() else {}
    xgb_params = train_summary.get("xgboost_params", {})
    run_metadata = train_summary.get("run_metadata", {})
    return pd.DataFrame([
        {
            "status": "ok",
            "export_experiment_id": run_metadata.get("export_experiment_id"),
            "input_run_id": run_metadata.get("input_run_id"),
            "model_run_id": run_metadata.get("model_run_id"),
            "run_kind": run_metadata.get("run_kind"),
            "sampled": train_summary.get("sampled"),
            "sample_rows": train_summary.get("sample_rows"),
            "feature_count": train_summary.get("feature_count"),
            "categorical_feature_count": len(train_summary.get("categorical_feature_columns", [])),
            "train_rows": train_summary.get("train_rows"),
            "val_rows": train_summary.get("val_rows"),
            "train_positive_ratio": train_summary.get("train_positive_ratio"),
            "scale_pos_weight": train_summary.get("scale_pos_weight"),
            "best_iteration": train_summary.get("best_iteration"),
            "best_score": train_summary.get("best_score"),
            "threshold_strategy": threshold_payload.get("threshold_strategy"),
            "threshold": threshold_payload.get("threshold"),
            "n_estimators": xgb_params.get("n_estimators"),
            "max_depth": xgb_params.get("max_depth"),
            "learning_rate": xgb_params.get("learning_rate"),
            "enable_categorical": xgb_params.get("enable_categorical"),
            "training_time_sec": train_summary.get("training_time_sec"),
            "memory_mb": train_summary.get("memory_mb"),
        }
    ])


def _metric_row(name: str, path: Path) -> dict:
    if not path.exists():
        return {"split": name, "status": "missing", "path": str(path)}

    payload = _read_json(path)
    metrics = payload.get("metrics", {})
    label_info = payload.get("label_summary", {})
    score_info = payload.get("score_profile", {})
    runtime_sec = payload.get("runtime_sec", {})
    total_runtime = runtime_sec.get("total_validate_xgb", runtime_sec.get("total_test_xgb"))

    return {
        "split": payload.get("split", name),
        "status": "ok",
        "sampled": payload.get("sampled"),
        "feature_count": payload.get("feature_count"),
        "feature_columns_hash": payload.get("feature_columns_hash"),
        "threshold_strategy": payload.get("threshold_strategy"),
        "threshold": metrics.get("threshold", payload.get("threshold")),
        "f1": metrics.get("f1"),
        "recall": metrics.get("recall"),
        "precision": metrics.get("precision"),
        "average_precision": metrics.get("average_precision"),
        "tp": metrics.get("tp"),
        "fp": metrics.get("fp"),
        "fn": metrics.get("fn"),
        "tn": metrics.get("tn"),
        "positive_count": label_info.get("positive_count"),
        "positive_ratio": label_info.get("positive_ratio"),
        "predicted_positive_count": score_info.get("predicted_positive_count"),
        "predicted_positive_rate": score_info.get("predicted_positive_rate"),
        "memory_mb": payload.get("memory_mb"),
        "total_runtime_sec": total_runtime,
        "path": str(path),
    }


artifact_paths = _artifact_paths(OUT_DIR)
artifact_status = pd.DataFrame([
    {"artifact": name, "exists": path.exists(), "path": str(path)}
    for name, path in artifact_paths.items()
])

feature_summary_df, feature_alignment_df = _build_feature_summary(artifact_paths)
run_summary_df = _build_run_summary(artifact_paths)
metric_summary_df = pd.DataFrame([
    _metric_row("val", artifact_paths["metrics_val"]),
    _metric_row("test", artifact_paths["metrics_test"]),
])
if "threshold_strategy" in metric_summary_df.columns:
    metric_summary_df["threshold_strategy"] = metric_summary_df["threshold_strategy"].astype("object")
if artifact_paths["threshold"].exists() and "threshold_strategy" in metric_summary_df.columns:
    threshold_payload = _read_json(artifact_paths["threshold"])
    metric_summary_df.loc[metric_summary_df["split"] == "val", "threshold_strategy"] = threshold_payload.get("threshold_strategy")

print("OUT_DIR:", OUT_DIR)
print("Actual trained feature count:", len(feature_summary_df))
display(artifact_status)

print("Train/threshold run summary")
display(run_summary_df)

print("Actual trained feature list and metadata")
display(feature_summary_df)

if feature_alignment_df.empty:
    print("No manifest/approve feature mismatch against trained feature_columns.json.")
else:
    print("Features present in manifest or approve contract but not in trained feature_columns.json")
    display(feature_alignment_df)

print("Validation / final test metrics from saved artifacts")
display(metric_summary_df)

if not artifact_paths["metrics_test"].exists():
    print("metrics_test.json not found: final test has not been evaluated for this OUT_DIR.")


OUT_DIR: /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_02_xgb_native_cat_15f/full/default_00
Actual trained feature count: 14


,artifact,exists,path
0,model,True,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_...
1,feature_columns,True,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_...
2,train_summary,True,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_...
3,feature_importance,True,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_...
4,threshold,True,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_...
5,metrics_val,True,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_...
6,confusion_matrix_val,True,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_...
7,metrics_test,True,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_...
8,confusion_matrix_test,True,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_...


Train/threshold run summary


,status,export_experiment_id,input_run_id,model_run_id,run_kind,sampled,sample_rows,feature_count,categorical_feature_count,train_rows,...,best_iteration,best_score,threshold_strategy,threshold,n_estimators,max_depth,learning_rate,enable_categorical,training_time_sec,memory_mb
0,ok,ml_00_test,run_02_xgb_native_cat_15f,default_00,full,False,None,14,7,3046861,...,179,0.051826,max_f1,0.919096,300,4,0.05,True,65.358945,3836.289062


Actual trained feature list and metadata


,feature_order,column_name,used_in_ml,feature_group,dtype,xgb_feature_type,encoding,source_column,excluded_reason,leakage_risk,selection_note,rank_by_gain,importance_gain,importance_weight,importance_cover
0,1,amount__current__value,TRUE,amount,float64,q,passthrough,amount__current__value,NaN,low_current_row_only,NaN,10,1225.928223,312.0,27286.548828
1,2,amount__current__log1p,TRUE,amount,float64,q,passthrough,amount__current__log1p,NaN,low_current_row_only,NaN,11,856.973328,45.0,41060.062500
2,3,amount_received__current__value,TRUE,amount,float64,q,passthrough,amount_received__current__value,add_candidate: 기본 False. parquet에 있으면 True로 변경 가능,low_current_row_only,NaN,6,11813.574219,220.0,97081.015625
3,4,amount_received__current__log1p,TRUE,amount,float64,q,passthrough,amount_received__current__log1p,add_candidate: 기본 False. parquet에 있으면 True로 변경 가능,low_current_row_only,NaN,8,3521.911621,28.0,40858.781250
4,5,amount__paid_recv_ratio,TRUE,amount,float64,q,passthrough,amount__paid_recv_ratio,add_candidate: 기본 False. parquet에 있으면 True로 변경 가능,low_current_row_only,NaN,9,2070.685303,15.0,40333.917969
5,6,cat__from_bank__xgb_cat,TRUE,categorical_xgb_native,category,c,xgb_native,From Bank,NaN,low_if_train_only_category_manifest,xgb_native encoded from raw/category source,2,40205.839844,357.0,152256.375000
6,7,cat__to_bank__xgb_cat,TRUE,categorical_xgb_native,category,c,xgb_native,To Bank,NaN,low_if_train_only_category_manifest,xgb_native encoded from raw/category source,7,11205.860352,673.0,228672.093750
7,8,cat__bank_pair__xgb_cat,TRUE,categorical_xgb_native,category,c,xgb_native,cat__bank_pair__code,NaN,low_if_train_only_category_manifest,xgb_native encoded from raw/category source,3,15766.884766,653.0,186692.062500
8,9,cat__payment_currency__xgb_cat,TRUE,categorical_xgb_native,category,c,xgb_native,Payment Currency,NaN,low_if_train_only_category_manifest,xgb_native encoded from raw/category source,12,373.319763,20.0,47426.117188
9,10,cat__receiving_currency__xgb_cat,TRUE,categorical_xgb_native,category,c,xgb_native,Receiving Currency,NaN,low_if_train_only_category_manifest,xgb_native encoded from raw/category source,13,2.882446,3.0,64.605598


Features present in manifest or approve contract but not in trained feature_columns.json


,column_name,source,used_in_approve_contract,used_in_trained_model
0,time__row__hour,manifest_only,False,False


Validation / final test metrics from saved artifacts


,split,status,sampled,feature_count,feature_columns_hash,threshold_strategy,threshold,f1,recall,precision,...,fp,fn,tn,positive_count,positive_ratio,predicted_positive_count,predicted_positive_rate,memory_mb,total_runtime_sec,path
0,val,ok,False,14,4759c2b7d9db2e361f78cbba956bc5fd2fcc8639d4029e...,max_f1,0.919096,0.116920,0.113573,0.120470,...,898,960,1013939,1083,0.001066,1021,0.001005,3646.765625,8.30882,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_...
1,test,ok,False,14,4759c2b7d9db2e361f78cbba956bc5fd2fcc8639d4029e...,max_f1,0.919096,0.105785,0.166388,0.077541,...,3557,1498,1010210,1797,0.001769,3856,0.003797,3740.257812,7.58510,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_...
